# Clase 1 · NovaRetail: de los datos crudos a un número en el que se pueda confiar

**La situación.** El equipo comercial de NovaRetail (una empresa ficticia que vende en tienda, web, app y marketplace) pidió el **ticket promedio por canal**. Tres personas del equipo de datos lo calcularon por separado y dieron **tres números distintos**.

Ninguna se equivocó programando. El problema fue otro: **nadie podía reconstruir el camino desde el dato crudo hasta el número**.

**Cómo vamos a trabajar.** Cada paso tiene tres momentos: *qué vamos a hacer*, la celda de código (se corre con `Shift` + `Enter`) y *qué vimos*. No hace falta saber programar de antemano: el objetivo es entender **qué decisión se toma en cada línea**.

> Dos palabras para empezar: un **DataFrame** es una tabla (filas y columnas), y una **semilla** es el número que fija un sorteo, para que todos obtengamos lo mismo.

## Paso 0 · Preparamos las herramientas

Usamos tres librerías de Python, cada una con un trabajo:

| Librería | Para qué la usamos hoy |
| --- | --- |
| **NumPy** | Hacer cuentas y sorteos con números |
| **pandas** | Trabajar con tablas: filtrar, agrupar, promediar |
| **matplotlib** | Dibujar gráficos |

También imprimimos **las versiones**. No es decoración: es lo que te permite explicar, seis meses después, por qué un resultado cambió.

**Y un archivo más: `ds_helpers.py`.** Es un archivo de ayudas que armamos para la clase y que vive **en la misma carpeta que este notebook** (`data-scientist/`). Trae las «cajitas» interactivas (las predicciones, las pistas, los verificadores que te dicen si lo hiciste bien) para no llenar el notebook de código repetido. La línea `from ds_helpers import *` lo carga. Si alguna vez aparece `ModuleNotFoundError: No module named 'ds_helpers'`, es que el notebook se abrió sin ese archivo al lado.

In [ ]:
from ds_helpers import *            # trae NumPy (np), pandas (pd) y las ayudas de la clase
import sys
import matplotlib.pyplot as plt

print("python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy: ", np.__version__)

def barras(serie, titulo, ylabel="", horizontal=False, figsize=(7, 3.2)):
    """Un gráfico de barras simple, para no repetir código en cada paso."""
    fig, ax = plt.subplots(figsize=figsize)
    serie.plot(kind="barh" if horizontal else "bar", ax=ax, color="#7C5CF6")
    ax.set_title(titulo)
    if horizontal:
        ax.set_xlabel(ylabel)
    else:
        ax.set_ylabel(ylabel)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

**Qué vimos:** quedaron listas las herramientas y una pequeña función, `barras`, que usaremos varias veces para graficar.

### Equipos

Armamos los equipos de la clase (3 o 4 personas). **Rotar roles** cambia quién conduce, implementa, valida y presenta.

In [ ]:
equipos_interactivo()

## Paso 1 · Armamos los datos de NovaRetail

No cargamos ningún archivo: **los generamos con código**, así todos tenemos exactamente los mismos datos. Y los generamos **como llegan en la vida real: crudos, con errores**. Vamos a crear **dos tablas relacionadas**:

```
clientes_crudos  ──(id_cliente)──▶  transacciones_crudas
una fila por cliente                  una fila por compra
```

| Tabla | Qué representa |
| --- | --- |
| `clientes_crudos` | Una fila por cliente, **tal como llega**: quién es, cómo compra y si abandonó, con datos faltantes y repetidos |
| `transacciones_crudas` | Las compras **tal como llegan del sistema**: con errores de carga |

**Después, las limpiamos nosotros**, tomando y declarando cada decisión (Pasos 4 y 7). De ese trabajo salen dos tablas nuevas: **`transacciones`** y **`clientes`**, ya transacciones.

### 1.1 · La semilla y los parámetros

Casi todo lo que sigue es **un sorteo** (edades, canales, montos…). Para que el sorteo sea siempre el mismo usamos una **semilla**: con la misma semilla, cualquiera que corra este código obtiene exactamente los mismos datos.

In [ ]:
SEMILLA = 2025                                 # el «número de sorteo»
rng = np.random.default_rng(SEMILLA)           # el generador de sorteos
N_CLIENTES = 8000
FECHA_CORTE = pd.Timestamp("2025-12-31")       # el «hoy» de nuestro caso
print("Listo: vamos a generar", N_CLIENTES, "clientes")

### 1.2 · Quiénes son los clientes

Sorteamos para cada cliente su **canal principal**, su **segmento** (Esencial, Plus o Premium), su **región**, su **edad** y cuándo se dio de alta. Fijate un detalle: el segmento depende del canal. En la web hay más clientes del segmento Esencial, y eso va a importar más adelante.

In [ ]:
canal = rng.choice(["web", "app", "tienda"], size=N_CLIENTES, p=[0.40, 0.25, 0.35])
p_seg = {"web": [0.62, 0.28, 0.10], "app": [0.30, 0.45, 0.25], "tienda": [0.28, 0.42, 0.30]}
segmento = np.array([rng.choice(["Esencial", "Plus", "Premium"], p=p_seg[c]) for c in canal])
region = rng.choice(["Centro", "Norte", "Sur", "Litoral", "Cuyo"], size=N_CLIENTES, p=[0.30, 0.20, 0.18, 0.17, 0.15])
edad = np.clip(rng.normal(41, 12, N_CLIENTES), 18, 80).round().astype(int)
alta = pd.Timestamp("2021-01-01") + pd.to_timedelta(rng.integers(0, 1730, N_CLIENTES), unit="D")
antig = ((FECHA_CORTE - alta).days / 30.4).round().astype(int)       # meses desde el alta
print("canal, segmento, región, edad y alta: listos")

### 1.3 · Cómo compra cada cliente

Ahora su comportamiento: **cuántas compras** hizo en el último año, **cuánto gasta por compra** (`ticket`), **hace cuántos días** compró por última vez, cuántas veces **llamó a soporte**, si usa la app, qué **descuento** recibe y qué tan **satisfecho** está.

In [ ]:
base_compras = np.select([segmento == "Esencial", segmento == "Plus"], [5, 9], 14)
compras = rng.poisson(base_compras)
ticket = np.round(np.select([segmento == "Esencial", segmento == "Plus"], [35, 62], 110) * rng.lognormal(0, 0.25, N_CLIENTES), 2)
dias = np.clip(rng.gamma(2.2, 24, N_CLIENTES), 1, 365).round().astype(int)
soporte = rng.poisson(np.where(segmento == "Esencial", 1.2, 0.6))
usa_app = np.where(canal == "app", 1, (rng.random(N_CLIENTES) < 0.35).astype(int))
descuento = np.clip(rng.normal(np.where(segmento == "Esencial", 0.14, 0.08), 0.05, N_CLIENTES), 0, 0.35).round(3)
sat_real = np.clip(np.round(rng.normal(3.7 - 0.12 * soporte + 0.15 * (segmento == "Premium"), 0.9, N_CLIENTES)), 1, 5)
print("compras, ticket, recencia, soporte, app, descuento y satisfacción: listos")

### 1.4 · Quién abandona

Esta es la variable que queremos entender: **`abandono_90d`** (1 = el cliente se fue en los 90 días siguientes al corte). La armamos con una regla simple: la probabilidad de irse **sube** si hace mucho que no compra o llama seguido a soporte, y **baja** si usa la app o está satisfecho. Después se «sortea» quién se va.

In [ ]:
z_dias = (dias - 60) / 40
logit = (-2.45
         + 0.85 * (segmento == "Esencial") - 0.45 * (segmento == "Premium")
         + 0.80 * z_dias + 0.30 * soporte - 0.42 * (sat_real - 3.7)
         - 0.50 * usa_app - 0.012 * (antig - 24) - 0.04 * (compras - 8)
         + 0.55 * ((canal == "tienda") & (segmento == "Esencial")))
prob_abandono = 1 / (1 + np.exp(-logit))
abandono = (rng.random(N_CLIENTES) < prob_abandono).astype(int)
print(f"Abandonan {abandono.mean():.1%} de los clientes")

### 1.5 · Armamos la tabla `clientes_crudos` (con defectos a propósito)

Como en una base real, **no todo viene perfecto**. Rompemos algunas cosas a propósito para tener algo que detectar:

- A algunos clientes **les falta** la satisfacción, el ticket o la región. La satisfacción falta más entre quienes abandonan: **el faltante dice algo**.
- Algunas **edades son imposibles** (0, −1, 150).
- Agregamos `motivo_baja`, que **solo existe para quienes ya abandonaron**. Esto es una *fuga*: una variable que usa información del futuro.
- Duplicamos **60 filas**, como por un error de carga.

In [ ]:
sat = sat_real.copy()
sat[rng.random(N_CLIENTES) < np.where(abandono == 1, 0.30, 0.09)] = np.nan   # la satisfacción falta más entre quienes se van
ticket = ticket.astype(float)
ticket[rng.random(N_CLIENTES) < 0.03] = np.nan
region_s = pd.Series(region, dtype=object)
region_s[rng.random(N_CLIENTES) < 0.02] = np.nan
edad = edad.astype(float)
malos = rng.random(N_CLIENTES) < 0.004
edad[malos] = rng.choice([0, 150, -1], size=malos.sum())                      # edades imposibles
motivo = np.where(abandono == 1, rng.choice(["precio", "servicio", "competencia", "mudanza"], size=N_CLIENTES), None)

clientes_crudos = pd.DataFrame({
    "id_cliente": [f"NR{i:06d}" for i in range(1, N_CLIENTES + 1)],
    "fecha_alta": alta.strftime("%Y-%m-%d"), "canal": canal, "segmento": segmento, "region": region_s,
    "edad": edad, "antiguedad_meses": antig, "compras_12m": compras, "ticket_promedio": ticket,
    "dias_desde_ultima_compra": dias, "contactos_soporte_90d": soporte, "usa_app": usa_app,
    "descuento_promedio": descuento, "satisfaccion": sat, "motivo_baja": motivo, "abandono_90d": abandono,
})
repetidas = clientes_crudos.sample(60, random_state=SEMILLA)                  # 60 filas repetidas por «error de carga»
clientes_crudos = pd.concat([clientes_crudos, repetidas], ignore_index=True).sample(frac=1, random_state=SEMILLA).reset_index(drop=True)
clientes_crudos.head()

Esto es lo que significa cada columna de `clientes_crudos`:

In [ ]:
pd.DataFrame({
    "columna": ["id_cliente", "fecha_alta", "canal", "segmento", "region", "edad", "antiguedad_meses", "compras_12m",
                "ticket_promedio", "dias_desde_ultima_compra", "contactos_soporte_90d", "usa_app", "descuento_promedio",
                "satisfaccion", "motivo_baja", "abandono_90d"],
    "qué es": ["Identificador del cliente (debería ser único)", "Cuándo se dio de alta", "Canal principal: web, app o tienda",
               "Esencial, Plus o Premium", "Región (puede faltar)", "Edad en años (hay valores imposibles)", "Meses desde el alta",
               "Compras en los últimos 12 meses", "Importe promedio por compra (puede faltar)", "Días desde la última compra",
               "Veces que llamó a soporte en 90 días", "1 si usa la app", "Descuento promedio (de 0 a 1)",
               "Encuesta de 1 a 5 (falta más entre quienes se van)", "Por qué se fue: ¡solo existe si ya se fue!",
               "LO QUE QUEREMOS ENTENDER: 1 si abandonó en los 90 días siguientes"]})

**Qué vimos:** `clientes_crudos` tiene una fila por cliente (más 60 repetidas) y 16 columnas. Ya hay trampas sembradas, y las vamos a encontrar.

### 1.6 · Las compras: cómo habrían sido «en la realidad»

Ahora generamos **las compras**, una fila por compra. Para fabricar un archivo creíble, primero las generamos **como serían si todo estuviera perfecto**: a esa tabla la llamamos **`compras_reales`**. En la vida real nadie la tiene; **la guardamos solo para poder comprobar, al final, si nuestra limpieza llegó a lo correcto**.

Las construimos **a partir de los clientes**, para que todo sea coherente:

- cada cliente tiene **tantas compras como dice `compras_12m`**;
- su última compra cae a `dias_desde_ultima_compra` días del corte;
- el monto promedio de sus compras es su `ticket_promedio`.

Primero decidimos **cuántas** compras hay y **en qué fecha** ocurre cada una.

In [ ]:
rng_tx = np.random.default_rng(SEMILLA + 1)                 # un sorteo aparte para las transacciones
unicos = clientes_crudos.drop_duplicates(subset="id_cliente").reset_index(drop=True)   # un cliente, una fila
n_tx = unicos["compras_12m"].to_numpy().astype(int)         # cuántas compras tiene cada cliente
idx = np.repeat(np.arange(len(unicos)), n_tx)               # a qué cliente pertenece cada compra
total = len(idx)

ultima = FECHA_CORTE - pd.to_timedelta(unicos["dias_desde_ultima_compra"].to_numpy(), unit="D")   # fecha de la última compra
desde = np.maximum(pd.to_datetime(unicos["fecha_alta"]).to_numpy(), np.datetime64(FECHA_CORTE - pd.Timedelta(days=365)))
desde = np.where(desde > ultima.to_numpy(), ultima.to_numpy(), desde)
dias_rango = ((ultima.to_numpy() - desde) / np.timedelta64(1, "D")).astype(int) + 1
corrimiento = np.floor(rng_tx.random(total) * dias_rango[idx]).astype(int)
fecha = pd.Series(pd.to_datetime(desde[idx]) + pd.to_timedelta(corrimiento, unit="D"))
es_la_ultima = np.concatenate([[True], idx[1:] != idx[:-1]])
fecha[es_la_ultima] = pd.to_datetime(ultima.to_numpy()[idx[es_la_ultima]])      # la última compra coincide con el dato del cliente
print(f"{total:,} compras en total, repartidas entre {len(unicos):,} clientes")

Después decidimos **qué se compró**: el monto, el canal, la categoría, el descuento y si el cliente **devolvió** el producto.

In [ ]:
crudo = rng_tx.gamma(shape=3.0, scale=40.0, size=total)
ref = unicos["ticket_promedio"].fillna(unicos["segmento"].map({"Esencial": 35.0, "Plus": 62.0, "Premium": 110.0})).to_numpy()
media = pd.Series(crudo).groupby(idx).transform("mean").to_numpy()
monto = np.round(crudo / media * ref[idx], 2)               # el promedio de cada cliente queda igual a su ticket_promedio

principal = unicos["canal"].to_numpy()[idx]
usa_app_tx = unicos["usa_app"].to_numpy()[idx]
alternativo = np.array(["web", "tienda", "marketplace", "app"])[rng_tx.integers(0, 4, size=total)]
alternativo = np.where((alternativo == "app") & (usa_app_tx == 0), "marketplace", alternativo)
canal_tx = np.where(rng_tx.random(total) < 0.85, principal, alternativo)        # casi siempre compra por su canal principal

PESOS_CATEGORIA = {"Esencial": [0.25, 0.15, 0.20, 0.40], "Plus": [0.25, 0.25, 0.25, 0.25], "Premium": [0.25, 0.40, 0.25, 0.10]}
CATEGORIAS = ["hogar", "tecnologia", "indumentaria", "alimentos"]
seg_tx = unicos["segmento"].to_numpy()[idx]
categoria = np.empty(total, dtype=object)
for s, p in PESOS_CATEGORIA.items():
    m = seg_tx == s
    categoria[m] = rng_tx.choice(CATEGORIAS, size=int(m.sum()), p=p)

descuento_tx = np.round(np.clip(rng_tx.normal(unicos["descuento_promedio"].to_numpy()[idx], 0.04), 0, 0.35), 2)
devuelta = rng_tx.random(total) < 0.08                      # alrededor del 8 % se devuelve

compras_reales = pd.DataFrame({"id_cliente": unicos["id_cliente"].to_numpy()[idx], "fecha": fecha.to_numpy(), "canal": canal_tx,
                               "categoria": categoria, "monto": monto, "descuento": descuento_tx, "devuelta": devuelta})
compras_reales = compras_reales.sort_values(["fecha", "id_cliente"], kind="stable").reset_index(drop=True)
compras_reales.insert(0, "id_transaccion", [f"T{i:06d}" for i in range(1, len(compras_reales) + 1)])
compras_reales.head()

**Qué vimos:** `compras_reales` es cómo serían las compras **sin ningún error**: fecha de verdad, un canal escrito de una sola forma y el monto siempre completo. Todavía **nadie más la ve**: ahora fabricamos el archivo que sí llega.

### 1.7 · La tabla `transacciones_crudas`: cómo llegan los datos de verdad

En la vida real, un archivo de transacciones **no llega clientes**. Para simularlo, tomamos `compras_reales` y le «estropeamos» cosas a propósito:

- la **fecha** pasa a ser texto;
- el **canal** se escribe de varias formas (`tienda`, `Tienda`, `TIENDA `, `ecommerce`…);
- faltan algunos **montos** y **descuentos**;
- algunos **montos son absurdos** (un cero de más, multiplicado por 500);
- hay **filas repetidas**;
- «devuelta» pasa a ser una columna `tipo` (venta o devolución).

In [ ]:
rng_crudas = np.random.default_rng(SEMILLA + 2)
VARIANTES_CANAL = {"tienda": ["tienda", "Tienda", "TIENDA "], "web": ["web", "Web ", "ecommerce"],
                   "app": ["app", "APP", "aplicacion"], "marketplace": ["marketplace", "Marketplace"]}

d = compras_reales.copy()
d["fecha"] = d["fecha"].dt.strftime("%Y-%m-%d")                            # la fecha llega como texto
d["canal"] = [VARIANTES_CANAL[c][0] if rng_crudas.random() < 0.70 else VARIANTES_CANAL[c][int(rng_crudas.integers(1, len(VARIANTES_CANAL[c])))]
              for c in d["canal"]]                                          # el canal, escrito de varias formas
d["tipo"] = np.where(d.pop("devuelta"), "devolucion", "venta")
monto_c = d["monto"].to_numpy().copy()
monto_c[rng_crudas.random(len(d)) < 0.04] = np.nan                          # 4 % de montos faltantes
atipico = rng_crudas.random(len(d)) < 0.001
monto_c[atipico & ~np.isnan(monto_c)] *= 500                                # 0,1 % de montos absurdos
d["monto"] = monto_c
desc = d["descuento"].to_numpy().copy()
desc[rng_crudas.random(len(d)) < 0.06] = np.nan                             # 6 % de descuentos faltantes
d["descuento"] = desc
repetidas_tx = d.sample(frac=0.003, random_state=SEMILLA)                    # 0,3 % de filas repetidas
d = pd.concat([d, repetidas_tx]).sort_values("id_transaccion", kind="stable").reset_index(drop=True)
transacciones_crudas = d[["id_transaccion", "id_cliente", "fecha", "canal", "tipo", "categoria", "monto", "descuento"]]
transacciones_crudas.head()

Resumen de las tablas que acabamos de armar. **Las dos que vamos a usar son las crudas**:

In [ ]:
pd.DataFrame({
    "tabla": ["clientes_crudos", "transacciones_crudas", "compras_reales"],
    "filas": [len(clientes_crudos), len(transacciones_crudas), len(compras_reales)],
    "una fila es…": ["un cliente, tal como llega (con 60 repetidas)", "una compra, tal como llega del sistema", "una compra, sin ningún error"],
    "para qué la usamos": ["la vamos a limpiar → `clientes`", "la vamos a limpiar → `transacciones`", "solo para comprobar al final"]})

Por último, le avisamos a las ayudas de la clase cuáles son nuestros datos (para que los verificadores trabajen con ellos) y le damos un **nombre corto** al archivo crudo, que vamos a usar en los próximos pasos:

In [ ]:
usar_datos(clientes_crudos)
crudas = transacciones_crudas      # nombre corto para el archivo crudo

## Paso 2 · Antes de calcular nada: los cinco controles

Ahora sí, el problema de NovaRetail. Antes de calcular **cualquier** número de negocio, hacemos cinco chequeos sobre el archivo tal como llegó. Son cinco preguntas simples:

| Control | La pregunta |
| --- | --- |
| 1. Forma | ¿Cuántas filas y columnas hay? ¿Es lo esperado? |
| 2. Tipos | ¿Cada columna es del tipo correcto? (la fecha, ¿es fecha? el monto, ¿es número?) |
| 3. Nulos | ¿Dónde faltan datos? |
| 4. Unicidad | ¿La clave (`id_transaccion`) es realmente única? |
| 5. Rangos | ¿Los valores tienen sentido? ¿Hay un canal que no existe, o montos absurdos? |

In [ ]:
crudas = transacciones_crudas        # el archivo tal como llegó

print("1. Forma:", crudas.shape, "(filas, columnas)")
print("2. Tipos:\n", crudas.dtypes, sep="")
print("3. Nulos por columna:\n", crudas.isna().sum(), sep="")
print("4. IDs repetidos:", crudas["id_transaccion"].duplicated().sum())
print("5. Canales que aparecen:", sorted(crudas["canal"].unique()))
print("   Monto mínimo:", crudas["monto"].min(), "· máximo:", crudas["monto"].max(), "· mediana:", crudas["monto"].median())

**Qué vimos:** los cinco controles encontraron problemas. La fecha es texto (`str`), faltan montos y descuentos, hay IDs repetidos, el canal aparece escrito de muchas formas y hay un monto máximo muy lejos de la mediana. Miremos dos de esos problemas en un gráfico.

**El canal escrito de muchas formas.** Cada barra es una «forma de escribirlo». Para una persona es el mismo canal; para la computadora, son canales distintos.

In [ ]:
barras(crudas["canal"].value_counts(), "Compras por cómo está escrito el canal", "compras")

**Los montos.** La mayoría de las compras está cerca de la mediana. Las pocas barras lejanas a la derecha son los montos absurdos (el eje está en escala logarítmica para poder verlos).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(crudas["monto"].dropna(), bins=np.logspace(0, np.log10(crudas["monto"].max()), 50), color="#7C5CF6")
ax.set_xscale("log")
ax.set_xlabel("monto (escala logarítmica)")
ax.set_ylabel("compras")
ax.set_title("Distribución de los montos en el archivo crudo")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

> **Regla de oro:** no se calcula nada de negocio antes de pasar estos cinco controles. Un promedio sobre datos mal cargados es un número con apariencia de verdad.

## Paso 3 · ¿Qué pasa si calculamos el ticket promedio igual?

Hagamos lo que haría una persona con apuro: **promediar el monto por canal, directamente sobre el archivo crudo**. Antes de correrlo, una predicción.

In [ ]:
predecir(
    "Calculamos el ticket promedio por canal directamente sobre el archivo crudo. ¿Qué sale?",
    ["Una fila por canal, con promedios razonables", "Muchas filas por canal y promedios distorsionados"],
    "Muchas filas por canal y promedios distorsionados",
    "El mismo canal aparece con varias escrituras, y los montos absurdos y repetidos inflan cada promedio. Es un número con apariencia de verdad.",
)

In [ ]:
sin_limpiar = crudas.groupby("canal")["monto"].agg(transacciones="count", ticket_promedio="mean").round(2)
sin_limpiar

**Qué vimos:** en lugar de 4 canales salen más de 10 filas, y los promedios son altísimos porque los montos absurdos arrastran la cuenta. Lo graficamos:

In [ ]:
barras(sin_limpiar["ticket_promedio"].sort_values(), "Ticket promedio SIN limpiar (por cómo está escrito el canal)", "ticket promedio", horizontal=True, figsize=(7, 3.6))

## Paso 4 · Declaramos nuestras decisiones, una por una

Ahora arreglamos el archivo. Lo importante: **cada arreglo es una decisión, y la escribimos**. Eso es lo que las tres personas de NovaRetail no hicieron. Vamos de a una, mostrando el «antes y después».

### Decisión 1 · Un id, una fila

Si una compra aparece dos veces, se cuenta dos veces. Nos quedamos con una sola fila por `id_transaccion`.

In [ ]:
transacciones = crudas.drop_duplicates("id_transaccion").copy()
print(f"Filas: {len(crudas):,} → {len(transacciones):,}  (sacamos {len(crudas) - len(transacciones)} repetidas)")

### Decisión 2 · Un canal, una escritura

Pasamos todo a minúsculas, sacamos espacios y unificamos los alias (`ecommerce` es `web`; `aplicacion` es `app`).

In [ ]:
transacciones["canal"] = transacciones["canal"].str.strip().str.lower().replace({"ecommerce": "web", "aplicacion": "app"})
print("Canales ahora:", sorted(transacciones["canal"].unique()))

### Decisión 3 · La fecha, como fecha

Convertimos el texto en una fecha de verdad, para poder filtrar y ordenar por tiempo.

In [ ]:
transacciones["fecha"] = pd.to_datetime(transacciones["fecha"])
print("Tipo de la columna fecha:", transacciones["fecha"].dtype)

### Decisión 4 · ¿Qué es un monto «atípico»?

Algunos montos son absurdos (un cero de más). Pocos, pero **un promedio es muy sensible a los extremos**. Necesitamos una regla para dejarlos afuera.

Usamos esta: **atípico = más de 10 veces el monto típico**. Para el «monto típico» usamos la **mediana** (el valor del medio) y no el promedio, porque la mediana **no se contamina con los extremos**. La regla de «10 veces» es una decisión nuestra: otra persona podría elegir otra y estaría bien, mientras la **declare**.

In [ ]:
umbral = 10 * transacciones["monto"].median()
print(f"Monto típico (mediana): {transacciones['monto'].median():.2f}  →  umbral de atípicos: {umbral:.2f}")
print("Compras por encima del umbral:", (transacciones["monto"] > umbral).sum())

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(transacciones["monto"].dropna(), bins=np.logspace(0, np.log10(transacciones["monto"].max()), 50), color="#7C5CF6")
ax.axvline(umbral, color="#F59E0B", lw=2, label=f"umbral = {umbral:.0f}")
ax.set_xscale("log"); ax.set_xlabel("monto (escala logarítmica)"); ax.set_ylabel("compras"); ax.legend()
ax.set_title("Lo que queda a la derecha de la línea se considera atípico")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

### Decisión 5 · ¿Qué compras entran al ticket promedio?

Solo las **ventas** (sin devoluciones), con **monto completo** y **no atípico**. Con eso, ahora sí calculamos el ticket por canal.

In [ ]:
validas = transacciones[transacciones["monto"].notna() & (transacciones["monto"] <= umbral) & (transacciones["tipo"] == "venta")]
print(f"Compras que entran al cálculo: {len(validas):,} de {len(transacciones):,}")

ticket = (validas.groupby("canal")["monto"].agg(transacciones="count", ticket_promedio="mean").round(2)
                 .sort_values("ticket_promedio", ascending=False))
ticket

In [ ]:
barras(ticket["ticket_promedio"], "Ticket promedio por canal (con las decisiones declaradas)", "ticket promedio")

**Qué vimos:** ahora hay **4 canales** y los montos son razonables. Cada número se puede explicar con una de las cinco decisiones. Y ya tenemos la tabla **`transacciones`**, limpia.

## Paso 5 · ¿Llegamos a lo correcto?

Tenemos algo que en la vida real casi nunca existe: **las compras reales** (`compras_reales`), la tabla sin errores que generamos al principio. Comparemos nuestro resultado contra ella. Si el camino que declaramos es bueno, los números tienen que parecerse.

In [ ]:
referencia = compras_reales[~compras_reales["devuelta"]].groupby("canal")["monto"].mean().round(2)
comparacion = pd.DataFrame({"ticket_declarado": ticket["ticket_promedio"], "compras_reales": referencia})
comparacion["diferencia"] = (comparacion["ticket_declarado"] - comparacion["compras_reales"]).round(2)
comparacion

In [ ]:
comparacion[["ticket_declarado", "compras_reales"]].plot(kind="bar", figsize=(7, 3.2), color=["#7C5CF6", "#10D9A0"])
plt.title("Nuestro ticket contra las compras reales"); plt.ylabel("ticket promedio"); plt.xticks(rotation=0); plt.ylim(0, 110); plt.tight_layout(); plt.show()

**Qué vimos:** la diferencia es mínima, y se explica: descartamos los montos faltantes. **Documentar esa decisión** es lo que hace el camino recuperable.

## Paso 6 · Por qué tres personas dieron tres números distintos

Con la misma tabla, tres decisiones **implícitas** dan tres números. Las dos que podemos mostrar con estos datos:

1. **¿Qué se hace con las devoluciones?** ¿Se incluyen o se excluyen?
2. **¿Se trabajó sobre una muestra al azar?** Y si es así, ¿se anotó cómo se generó? Ahí entra la **semilla**.

(Las otras dos causas del caso, usar fecha de compra o de facturación y no registrar el entorno, se evitan igual: declarando qué fecha se usa e imprimiendo las versiones, como hicimos en el Paso 0.)

**Primero, la muestra.** `sample(frac=0.5)` toma la mitad de las ventas **al azar**. Corré la celda de abajo **dos o tres veces** y mirá la columna sin semilla.

In [ ]:
base = transacciones[transacciones["monto"].notna() & (transacciones["monto"] <= umbral)]
ventas = base[base["tipo"] == "venta"]

pd.DataFrame({
    "muestra_sin_semilla": ventas.sample(frac=0.5).groupby("canal")["monto"].mean(),                              # cambia en cada corrida
    "muestra_con_semilla_42": ventas.sample(frac=0.5, random_state=42).groupby("canal")["monto"].mean(),          # siempre igual
}).round(2)

**Qué vimos:** sin semilla, el resultado cambia en cada corrida, aunque el código no cambie. Con semilla, siempre es el mismo. **La semilla es el registro de cómo se generó la muestra.**

**Ahora todo junto**, con las devoluciones también:

In [ ]:
tres_numeros = pd.DataFrame({
    "excluyendo_devoluciones": ventas.groupby("canal")["monto"].mean(),
    "incluyendo_devoluciones": base.groupby("canal")["monto"].mean(),
    "muestra_sin_semilla": ventas.sample(frac=0.5).groupby("canal")["monto"].mean(),
    "muestra_con_semilla_42": ventas.sample(frac=0.5, random_state=42).groupby("canal")["monto"].mean(),
}).round(2)
tres_numeros.plot(kind="bar", figsize=(8, 3.4), color=["#7C5CF6", "#A48AFB", "#F59E0B", "#10D9A0"])
plt.title("El mismo ticket, cuatro formas de calcularlo"); plt.ylabel("ticket promedio"); plt.xticks(rotation=0)
plt.ylim(0, 115); plt.legend(fontsize=8, ncol=2, loc="upper center"); plt.tight_layout(); plt.show()
tres_numeros

**Conclusión del caso:** ninguna persona se equivocó. Cada una tomó decisiones que no escribió. **Declarar cada decisión, fijar la semilla y registrar el entorno** es lo que vuelve el número recuperable.

## Paso 7 · Ahora los clientes: ¿cuándo podemos decir que un grupo abandona más?

Aplicamos la misma idea a otra pregunta: **¿qué grupo de clientes presenta mayor riesgo de abandono?** Pero antes de afirmar nada, hay que revisar la tabla `clientes_crudos`, igual que revisamos las transacciones.

### 7.1 · Primer vistazo

In [ ]:
clientes_crudos.head()

In [ ]:
print("Filas y columnas:", clientes_crudos.shape)
clientes_crudos.dtypes.to_frame("tipo")

### 7.2 · ¿`id_cliente` es una clave única?

Cada cliente debería aparecer una sola vez. Antes de mirar, una predicción.

In [ ]:
predecir(
    "¿Es id_cliente una clave única, un cliente por fila?",
    ["Sí, un cliente por fila", "No, hay clientes repetidos"],
    "No, hay clientes repetidos",
    "Una clave que se repite duplica clientes y distorsiona cualquier tasa. Siempre se verifica antes de agrupar.",
)

In [ ]:
print(f"id_cliente único: {clientes_crudos.id_cliente.is_unique} · filas repetidas: {clientes_crudos.duplicated('id_cliente').sum()}")

### 7.3 · ¿Dónde faltan datos?

In [ ]:
faltantes = clientes_crudos.isna().mean().sort_values(ascending=False)
barras(faltantes[faltantes > 0] * 100, "Porcentaje de faltantes por columna", "% de filas")

**Qué vimos:** falta la **satisfacción**, el **ticket**, la **región**. Y `motivo_baja` falta en la gran mayoría: es normal, porque **solo existe para quienes ya abandonaron**. Esa columna es una trampa: guardala en la cabeza.

### 7.4 · ¿Cuánto abandona, en general?

In [ ]:
tasa_base = clientes_crudos.drop_duplicates("id_cliente")["abandono_90d"].mean()
print(f"Abandonan el {tasa_base:.1%} de los clientes. Esta es la «tasa base»: la referencia contra la que vamos a comparar todo.")

### 7.5 · Una limpieza incorrecta: «eliminemos todos los faltantes»

Es lo primero que se le ocurre a mucha gente: borrar todas las filas con algún dato faltante. Veamos qué pasa.

In [ ]:
predecir(
    f"La tabla tiene {len(clientes_crudos):,} filas. Hacemos dropna(). ¿Cuántas sobreviven?",
    ["Casi todas", "Alrededor de la mitad", "Muy pocas (menos del 15 %)"],
    "Muy pocas (menos del 15 %)",
    "motivo_baja solo está completa para quienes abandonaron: dropna() se queda casi solo con ellos.",
)

In [ ]:
incorrecto = clientes_crudos.dropna()
print(f"Antes: {len(clientes_crudos):,} filas · después de dropna(): {len(incorrecto):,} ({len(incorrecto) / len(clientes_crudos):.0%})")

In [ ]:
predecir(
    "¿Qué pasa con la tasa de abandono de las filas que quedaron?",
    ["Se mantiene cerca del 14 %", "Baja", "Sube a casi el 100 %"],
    "Sube a casi el 100 %",
    "Nos quedamos solo con quienes tienen motivo de baja: es decir, con quienes ya abandonaron. Un artefacto, no un hallazgo.",
)

In [ ]:
print(f"Tasa de abandono tras dropna(): {incorrecto['abandono_90d'].mean():.1%}   (antes: {tasa_base:.1%})")

**(Opcional)** Probemos algo menos extremo: ignorar `motivo_baja` y borrar las filas con otros faltantes.

In [ ]:
otras = [col for col in clientes_crudos.columns if col != "motivo_baja"]
parcial = clientes_crudos.dropna(subset=otras)
print(f"Filas: {len(parcial):,} de {len(clientes_crudos):,} · tasa de abandono: {parcial['abandono_90d'].mean():.1%}   (antes: {tasa_base:.1%})")

**Qué vimos:** borrar sin pensar **cambia el resultado**: en el caso extremo la tasa se va a casi 100 %, y en el moderado baja, porque la satisfacción falta más entre quienes abandonan. Borrar filas **no es neutro**.

### 7.6 · Una decisión por columna, y comprobamos que no cambiamos nada importante

En vez de borrar a ciegas, decidimos columna por columna:

| Columna | Defecto | Decisión |
| --- | --- | --- |
| `id_cliente` | filas repetidas | quedarse con una fila por cliente |
| `edad` | valores imposibles | los que están fuera de 18–100 pasan a «faltante» |
| `region` | faltante | una categoría explícita: «Sin dato» |
| `satisfaccion` | falta más entre quienes se van | **no se rellena**: se agrega un indicador de «faltó» |
| `motivo_baja` | solo existe después de abandonar | **no se usa**: es fuga |

In [ ]:
clientes = clientes_crudos.drop_duplicates(subset="id_cliente").copy()                # un cliente, una fila
clientes["edad"] = clientes["edad"].where(clientes["edad"].between(18, 100))                # edades imposibles → faltante
clientes["region"] = clientes["region"].fillna("Sin dato")                                # región faltante → categoría explícita
clientes["satisfaccion_faltante"] = clientes["satisfaccion"].isna().astype(int)           # el faltante informa: se marca, no se rellena
# motivo_baja NO entra como variable: solo existe después del abandono (fuga)
print(f"Clientes: {len(clientes_crudos):,} → {len(clientes):,}")

**El control:** ¿conservamos a los clientes y la tasa de abandono?

In [ ]:
pd.DataFrame({
    "tabla": ["cruda", "dropna() sobre todo", "decisiones documentadas"],
    "filas": [len(clientes_crudos), len(incorrecto), len(clientes)],
    "tasa_abandono": [tasa_base, incorrecto["abandono_90d"].mean(), clientes["abandono_90d"].mean()],
}).round(3)

In [ ]:
verificar_universo(clientes)

### 7.7 · Una mirada rápida al abandono por canal

Con la tabla `clientes` ya limpia, miramos **cuánto abandona cada canal**.

In [ ]:
por_canal = clientes.groupby("canal")["abandono_90d"].agg(clientes="size", tasa_abandono="mean").round(3)
barras(por_canal["tasa_abandono"] * 100, "Abandono por canal", "% que abandona")
por_canal

**Qué vemos:** la web parece el canal más riesgoso. ¿Podemos afirmar que **el canal web provoca el abandono**? Esa es justamente la pregunta de hoy, y la vamos a contestar en el desafío. Una pista: ¿recordás que en la web había más clientes del segmento Esencial?

## Práctica guiada · Documentamos las decisiones

**Lo que hacen ahora, en equipo:** una **tabla de auditoría** (para cada variable con problemas: cuál es el defecto, qué decisión se toma y cómo se comprueba) y **dos decisiones en pandas** que no cambien el universo por accidente.

**Cómo se hace:**
1. En la tabla de abajo, la primera fila está **resuelta como ejemplo** (`id_cliente`). Completen las demás con la misma lógica.
2. Corran la celda: el verificador les dice qué falta.
3. Después, escriban dos decisiones en pandas y controlen que el universo se conserve.

In [ ]:
auditoria = pd.DataFrame([
    # EJEMPLO RESUELTO: así se completa una fila
    {"variable": "id_cliente", "defecto": "60 clientes aparecen dos veces",
     "decision": "quedarse con una fila por cliente (drop_duplicates)", "validacion": "id_cliente único y misma tasa de abandono"},
    # 👉 TU TURNO: completen defecto, decisión y validación de cada variable
    {"variable": "edad",         "defecto": "", "decision": "", "validacion": ""},
    {"variable": "region",       "defecto": "", "decision": "", "validacion": ""},
    {"variable": "satisfaccion", "defecto": "", "decision": "", "validacion": ""},
    {"variable": "motivo_baja",  "defecto": "", "decision": "", "validacion": ""},
])
verificar_auditoria(auditoria)

Ahora las **dos decisiones en pandas**. La primera está de ejemplo; la segunda es de ustedes: elijan una (edades imposibles, región faltante o satisfacción).

In [ ]:
equipo = clientes_crudos.copy()

# Decisión 1 · EJEMPLO RESUELTO: un cliente, una fila
equipo = equipo.drop_duplicates(subset="id_cliente")

# Decisión 2 · 👉 TU TURNO: escriban acá su decisión (por ejemplo, las edades fuera de 18–100 pasan a faltante)
equipo = ...

verificar_universo(equipo)      # ¿conservaron los clientes y la tasa de abandono?

In [ ]:
pistas([
    "Para cada variable, respondé tres preguntas: ¿qué está mal?, ¿qué decido hacer?, ¿cómo compruebo que no rompí nada?",
    "Una decisión sobre edad: .where(columna.between(18, 100)) deja como faltante lo que está fuera de rango.",
    "Para la región o la satisfacción, mirá cómo lo resolvimos en el Paso 7.6.",
    "Controlá siempre dos cosas: cuántas filas quedan y si cambió la tasa de abandono.",
])

## Puesta en común · Comparamos decisiones, no solo resultados

Dos equipos pueden tomar decisiones **distintas y válidas**. La pregunta es: *¿qué cambia en la conclusión y qué evidencia permitiría decidir entre ellas?*

Ejemplo con la satisfacción: el equipo A **marca** el faltante con un indicador; el equipo B lo **rellena con la mediana**.

In [ ]:
a = clientes.groupby("satisfaccion_faltante")["abandono_90d"].agg(clientes="size", tasa_abandono="mean").reset_index()
sat_imputada = clientes["satisfaccion"].fillna(clientes["satisfaccion"].median()).round()
b = clientes.assign(sat_imputada=sat_imputada).groupby("sat_imputada")["abandono_90d"].agg(clientes="size", tasa_abandono="mean").reset_index()
comparar_tablas(a, b, ("A: indicador de faltante", "B: imputación con la mediana"))

In [ ]:
predecir(
    "¿Qué se pierde en la opción B?",
    ["Nada: es equivalente", "La señal de que los clientes sin respuesta abandonan más", "Las filas con faltantes"],
    "La señal de que los clientes sin respuesta abandonan más",
    "Al imputar, los clientes sin respuesta parecen clientes comunes y se esconde que abandonan casi el triple.",
)

### Tres errores que se ven mucho

**Error 1 · Calcular algo con toda la tabla antes de separar entrenamiento y prueba** (se «filtra» información de la prueba al modelo).

In [ ]:
from sklearn.model_selection import train_test_split
X_tr, X_te = train_test_split(clientes, test_size=0.25, random_state=42, stratify=clientes["abandono_90d"])
media_total, media_train = clientes["satisfaccion"].mean(), X_tr["satisfaccion"].mean()
predecir(
    "Calculamos la media de satisfacción con toda la tabla y la usamos para rellenar. ¿Dónde está el problema?",
    ["No hay problema", "La prueba influyó en un valor que usa el modelo", "La media es incorrecta"],
    "La prueba influyó en un valor que usa el modelo",
    f"Media con todo: {media_total:.4f} · media solo con entrenamiento: {media_train:.4f}. Es poca diferencia, pero el principio es la fuga: todo lo que se aprende se ajusta solo con entrenamiento.",
)

**Error 2 · Confundir correlación con causa.** La web abandona más… pero ¿por la web, o por quién compra en la web?

In [ ]:
composicion = pd.crosstab(clientes["canal"], clientes["segmento"], normalize="index") * 100
composicion.plot(kind="bar", stacked=True, figsize=(7, 3.2), color=["#F59E0B", "#7C5CF6", "#10D9A0"])
plt.title("Qué segmentos compran en cada canal (%)"); plt.ylabel("% de clientes del canal"); plt.xticks(rotation=0)
plt.legend(title="segmento", loc="upper left", bbox_to_anchor=(1.0, 1.0)); plt.tight_layout(); plt.show()

In [ ]:
predecir(
    "El canal web tiene la tasa de abandono más alta. ¿Concluimos que el canal web provoca el abandono?",
    ["Sí, la tabla lo muestra", "No: el segmento Esencial pesa más en web y también abandona más", "No hay forma de saberlo"],
    "No: el segmento Esencial pesa más en web y también abandona más",
    "Parte de la diferencia es composición: web concentra más clientes Esencial. Un grupo de mayor riesgo exige controlar por las variables que lo confunden.",
)

**Error 3 · Usar una variable que usa información del futuro** (fuga). Armemos una variable con `motivo_baja` y veamos qué dice el verificador.

In [ ]:
fuga = clientes["motivo_baja"].notna().astype(int)
verificar_variable(fuga, "tiene_motivo_baja")

In [ ]:
predecir(
    "La variable «tiene motivo de baja» separa casi perfecto el abandono. ¿Es un gran predictor?",
    ["Sí, hay que incluirla", "No: solo existe después del abandono"],
    "No: solo existe después del abandono",
    "Una variable que no estaría disponible al momento de decidir es fuga: infla la métrica y no sirve en producción.",
)

## Pausa

Antes de salir, cada equipo deja escrito: **una decisión que confirmó**, **una duda** y **un riesgo** que ve. Las dudas las usamos para empezar el desafío.

In [ ]:
pausa_registro()

## Desafío por equipos · ¿Qué grupo abandona más, y qué podemos afirmar?

### Qué tienen que lograr
Un **mini análisis** de la tabla `clientes` (la que ya limpiamos) que responda: **¿qué combinación de canal y segmento abandona más?**, con:

| Entregable | Qué es |
| --- | --- |
| 1. Una **tabla** | Abandono por canal y segmento, con cuántos clientes hay en cada grupo |
| 2. Un **gráfico** | Un mapa de calor de esa tabla |
| 3. **Dos variables nuevas** | Con una justificación de negocio cada una |
| 4. Un **hallazgo** | Una frase, con su límite (qué *no* pueden afirmar) |

### Cómo trabajan
En equipo, con los roles: **conduce** (ordena), **implementa** (escribe el código), **valida** (controla) y **presenta** (cuenta). Cada paso tiene un ejemplo resuelto y un «tu turno». Yo voy a pasar preguntando, siempre lo mismo:
1. ¿Qué decisión habilita esto?
2. ¿Cuál es la población y el período?
3. ¿Cómo evitaron fuga o sesgo?
4. ¿Qué validaron?
5. ¿Qué no pueden afirmar?

### Paso A · La tabla
**Ejemplo resuelto: abandono por región.** Se agrupa por una columna, y para cada grupo se cuenta cuántos clientes hay y qué proporción abandona. Fijense que mostramos el **tamaño del grupo**: una tasa sin tamaño engaña (un 50 % sobre 2 clientes no es lo mismo que sobre 2.000).

In [ ]:
# EJEMPLO RESUELTO: abandono por región
(clientes.groupby("region")["abandono_90d"]
       .agg(clientes="size", tasa_abandono="mean")
       .reset_index().round(3))

**Tu turno:** la celda de abajo arranca con el ejemplo de la región. **Modifíquenla** para agrupar por **dos** columnas, `canal` y `segmento` (se pasan como lista). Las columnas finales tienen que llamarse `canal`, `segmento`, `clientes` y `tasa_abandono`.

In [ ]:
# 👉 TU TURNO: copiamos el ejemplo de arriba. Cambien "region" por las DOS columnas que piden (canal y segmento)
tabla = (clientes.groupby("region")["abandono_90d"]
               .agg(clientes="size", tasa_abandono="mean")
               .reset_index())

verificar_mini_analisis(tabla)      # el verificador les dice qué falta

### Paso B · El gráfico
Un **mapa de calor** muestra de un vistazo dónde está el problema: cuanto más oscuro, más abandono. Cada celda dice también el tamaño del grupo (`n`).

In [ ]:
heatmap_abandono(tabla)

### Paso C · Dos variables nuevas
Una **variable nueva** es una columna que construimos a partir de las que ya hay, para capturar algo del negocio. La regla: **tiene que responder a una pregunta de negocio** y **no puede usar información del futuro** (como el motivo de baja).

**Ejemplo resuelto:** `ticket_x_compras` = lo que gasta por compra × cuántas compras hizo. Mide cuánto aporta un cliente en el año. *Justificación: los clientes que aportan poco podrían ser los que más fácil se van.*

In [ ]:
# EJEMPLO RESUELTO
clientes["ticket_x_compras"] = clientes["ticket_promedio"] * clientes["compras_12m"]
verificar_variable(clientes["ticket_x_compras"], "ticket_x_compras")

**Tu turno:** creen **dos variables más**. Ideas: `compras_por_mes` (intensidad de compra), `recencia_alta` (hace más de 90 días que no compra), `usa_descuento_alto`. Escriban abajo, en un comentario, **por qué** cada una importa para el negocio.

In [ ]:
# 👉 TU TURNO · variable 1  (¿qué pregunta de negocio responde?)
clientes["var_1"] = ...
verificar_variable(clientes["var_1"], "var_1")

In [ ]:
# 👉 TU TURNO · variable 2
clientes["var_2"] = ...
verificar_variable(clientes["var_2"], "var_2")

### Paso D · El hallazgo
Escriban **una frase** con esta forma, y completen cada hueco con lo que vieron en su tabla:

> *«El grupo con mayor abandono es **[canal + segmento]** (**[X %]**, con **[n]** clientes). El canal **[…]** parece riesgoso en conjunto, pero **[por qué ese número engaña]**. Esto **describe dónde** se abandona más, **no demuestra por qué**.»*

Un buen hallazgo dice **dónde**, da el **tamaño del grupo** y declara su **límite**.

In [ ]:
pistas([
    "Trabajen siempre sobre la tabla `clientes` (la limpia), no sobre `clientes_crudos`.",
    "Para agrupar por dos columnas se pasa una lista: groupby([\"canal\", \"segmento\"]).",
    "Una variable nueva es una operación entre columnas: por ejemplo, clientes[\"compras_12m\"] / 12.",
    "Si verificar_variable les avisa de fuga, revisen que la variable no use `motivo_baja` ni `abandono_90d`.",
])

### Paso E · La evidencia
Al final, cada equipo guarda su trabajo en una **ficha de evidencia**. Abajo hay un **ejemplo ya completado** (sobre el análisis por región): reemplacen cada texto por el de su equipo.

In [ ]:
guardar_evidencia(
    equipo="Equipo 1",
    decision="Decidir en qué región conviene reforzar la retención primero.",            # ¿qué decisión habilita?
    poblacion="8.000 clientes únicos al 31/12/2025; abandono = baja en los 90 días siguientes.",
    procedimiento="Se limpió la tabla (sin repetidos, edades imposibles a faltante, motivo_baja afuera) y se agrupó por región.",
    resultado="La región con más abandono es X (Y %, con n clientes).",
    validacion="Se comprobó que la limpieza conservó los 8.000 clientes y la tasa de abandono.",
    limite="Describe dónde se abandona más; no prueba por qué ni qué efecto tendría una acción.",
)

## Defensa y revisión cruzada

Cada equipo presenta su hallazgo; otro equipo lo revisa y devuelve **una pregunta técnica, una de negocio y una sobre riesgo**, más **una fortaleza, una brecha prioritaria y una mejora que se pueda verificar**.

In [ ]:
revision_cruzada_form()

## Cierre

Volvemos a la pregunta de hoy: **¿qué controles hacen falta antes de afirmar que un grupo abandona más?** Cada persona completa su ticket de salida.

In [ ]:
cierre_individual()

In [ ]:
marcador()